# SAA+ Benchmark on MVTec and VisA

Reproduces image-AUROC / pixel-AUROC / AP / F1 from the SAA+ paper  
(~500 images per dataset, stratified sampling, T4 GPU)

In [ ]:
# Cai dat. An toan khi chay lai nhieu lan.
%cd /content

# Xoa clone cu TRUOC khi clone. Khong co dong nay thi git clone bao
# "destination path already exists", bo qua im lang, va ban chay tiep bang
# code cu ma khong biet.
!rm -rf /content/Segment-Any-Anomaly
!git clone -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/

# setuptools >= 80 da bo lenh `setup.py develop`, ma pip dung dung lenh do cho
# ban editable khi co --no-build-isolation. Colab nang image len la GroundingDINO
# gay voi "python setup.py develop did not run successfully". Ghim lai truoc.
!pip install -q "setuptools<80" wheel
import setuptools
print('setuptools:', setuptools.__version__)

# Go pin transformers<4.36 cua GroundingDINO. Phai quet CA requirements.txt,
# khong chi *.py: pip doc requirements.txt, va bo sot no thi pip ha transformers
# xuong 4.35 (keo theo huggingface_hub va tokenizers), roi lenh pip cuoi cell
# lai day len 5.x - vong xoay do de lai mot dong conflict gia.
# Code GroundingDINO trong repo nay da duoc va cho transformers 5.x tu truoc.
import re, pathlib

for pattern in ('*.py', '*.txt'):
    for p in pathlib.Path('GroundingDINO').rglob(pattern):
        txt = p.read_text()
        patched = re.sub(r'transformers[^"\'\n]*<4\.\d+(\.\d+)?', 'transformers>=4.41.0', txt)
        if patched != txt:
            p.write_text(patched)
            print('go pin transformers trong', p)

# KHONG dat -q cho hai lenh editable duoi day: day la cho de gay nhat, va
# loi that nam trong phan output ma -q nuot mat.
%cd GroundingDINO/
!pip install -e . --no-build-isolation
%cd ../SAM
!pip install -e .
%cd ..

!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru

# KHONG kiem tra import o day. Ban editable ghi duong dan vao mot file .pth,
# ma .pth chi duoc doc luc interpreter khoi dong - package vua cai xong van
# "khong ton tai" voi kernel dang chay. Kiem tra nam o cell sau lenh restart.
print('\nCai dat xong. Chay cell tiep theo de restart runtime,')
print('roi cell sau do se xac nhan package da cai duoc that.')

In [ ]:
# Restart so updated transformers is loaded from disk
import os
os.kill(os.getpid(), 9)

In [ ]:
# Xac nhan package da cai THAT - chay sau restart, vi ban editable chi hien
# ra voi interpreter khoi dong lai.
%cd /content/Segment-Any-Anomaly
import importlib.util

missing = [m for m in ('groundingdino', 'segment_anything')
           if importlib.util.find_spec(m) is None]

for m in ('groundingdino', 'segment_anything'):
    print(f'{m}: {"THIEU" if m in missing else "OK"}')

if missing:
    raise RuntimeError(
        f'Cai dat that bai: {missing}. DUNG chay tiep - moi class se chet o dong '
        f'import. Doc output pip cua cell 1 de biet la loi setuptools hay loi '
        f'bien dich CUDA extension.'
    )

%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

from google.colab import userdata
import json, pathlib, os

pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
pathlib.Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY')
}))
!chmod 600 /root/.kaggle/kaggle.json

# Accept dataset terms first at: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/datasets/ --unzip

os.environ['MVTEC_DIR'] = '/content/datasets'

from datasets import mvtec_classes
present = [c for c in mvtec_classes if os.path.isdir(f'/content/datasets/{c}')]
print(f'MVTec: {len(present)}/15 classes ready:', present)

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

!wget -q --show-progress \
    "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar" \
    -O /content/datasets/visa.tar
!tar -xf /content/datasets/visa.tar -C /content/datasets/

!python datasets/prepare_visa_public.py \
    --data-folder /content/datasets \
    --save-folder /content/datasets/VisA_pytorch \
    --split-file /content/datasets/split_csv/1cls.csv

import os
os.environ['VISA_DIR'] = '/content/datasets/VisA_pytorch/1cls'
print('VisA classes:', sorted(os.listdir(os.environ['VISA_DIR'])))

## Gan Google Drive lam noi luu ket qua

Ket qua ghi THANG vao Drive, tung class mot. Colab ngat giua chung thi nhung
class da xong van con, chay lai la resume tiep.

In [ ]:
# Ghi ket qua thang vao Drive. Moi class chay xong la an toan ngay - khong
# cho toi cuoi lan chay moi copy ra, vi disconnect la mat sach /content.
from google.colab import drive
import os, glob, json

drive.mount('/content/drive')

# Moi cau hinh mot thu muc rieng. Doi ten khi doi cau hinh (vd run_lite1),
# neu khong ket qua cau hinh sau ghi de len cau hinh truoc.
ROOT_DIR = '/content/drive/MyDrive/SAA_results/run_baseline'
os.environ['ROOT_DIR'] = ROOT_DIR
os.makedirs(f'{ROOT_DIR}/csv', exist_ok=True)

# Tat anh truc quan hoa: ~117 anh moi class, ghi qua Drive FUSE rat cham.
# Can anh cho luan van thi chay rieng mot class voi VIS=True sau.
os.environ['VIS'] = 'False'

print('ROOT_DIR =', ROOT_DIR)

csvs = glob.glob(f'{ROOT_DIR}/csv/*.csv')
meta_path = f'{ROOT_DIR}/csv/run_meta.json'
print('CSV da co   :', [os.path.basename(p) for p in csvs] or 'chua co')

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f'run_meta.json: {len(meta)} class - resume se bo qua chung neu cung cau hinh')
elif csvs:
    print('CANH BAO: co CSV nhung khong co run_meta.json.')
    print('Runner se coi nhung class do la khong dang tin va chay lai het.')
else:
    print('run_meta.json: chua co - lan chay dau tien')

In [ ]:
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['MVTEC_DIR'] = '/content/datasets'

# Spec muc 6.4: bat cal_pro cho cac cau hinh len bang chinh.
# r_f1 (max-F1-region) di chung co nay — tat co la cot r_f1 rong.
# CHOT GIA TRI NAY TRUOC KHI BAT DAU, dung doi giua chung:
# resume so sanh cal_pro trong run_meta.json, doi la chay lai het.
os.environ['CAL_PRO'] = 'True'
# os.environ['MAX_SAMPLES'] = '34'  # ~510 images across 15 classes

result = subprocess.run(
    ['python', 'run_MVTec.py'],
    cwd='/content/Segment-Any-Anomaly'
)
print('MVTec benchmark done, exit code:', result.returncode)

In [ ]:
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['VISA_DIR'] = '/content/datasets/VisA_pytorch/1cls'

# Spec muc 6.4: bat cal_pro cho cac cau hinh len bang chinh.
# r_f1 (max-F1-region) di chung co nay — tat co la cot r_f1 rong.
# CHOT GIA TRI NAY TRUOC KHI BAT DAU, dung doi giua chung:
# resume so sanh cal_pro trong run_meta.json, doi la chay lai het.
os.environ['CAL_PRO'] = 'True'
# os.environ['MAX_SAMPLES'] = '42'  # ~504 images across 12 classes

result = subprocess.run(
    ['python', 'run_VisA_public.py'],
    cwd='/content/Segment-Any-Anomaly'
)
print('VisA benchmark done, exit code:', result.returncode)

## Buoc 1 — profiling 4 cau hinh tren mot class

Chon ra cau hinh Lite thang cuoc. Can MobileSAM / EfficientViT-SAM da cai va checkpoint da tai.

In [ ]:
# Buoc 1 (spec muc 6.2) — profiling latency tren MOT class, 4 cau hinh.
# Goi thang eval_SAA.py, KHONG qua runner: moi cau hinh can --root-dir rieng,
# neu khong cau hinh sau ghi de hang 'carpet' cua cau hinh truoc.
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['MVTEC_DIR'] = '/content/datasets'

CONFIGS = [
    ('baseline', 'vit_h',           'wide_resnet50', 'weights/sam_vit_h_4b8939.pth'),
    ('lite1',    'mobile_sam',      'wide_resnet50', 'weights/mobile_sam.pt'),
    ('lite2',    'mobile_sam',      'mobilenetv3',   'weights/mobile_sam.pt'),
    ('lite3',    'efficientvit_l0', 'wide_resnet50', 'weights/efficientvit_sam_l0.pt'),
]

# Khong dat --max-samples: ca 4 cau hinh phai chay dung cung 117 anh thi p_ap
# va p_f1 moi so duoc voi nhau (rang buoc normalize(), spec muc 2.7).
for name, sam, saliency, ckpt in CONFIGS:
    if not os.path.exists(ckpt):
        print(f'skip {name}: thieu checkpoint {ckpt}')
        continue
    print(f'=== {name}: sam={sam} saliency={saliency} ===')
    r = subprocess.run([
        'python', 'eval_SAA.py',
        '--dataset', 'mvtec', '--class-name', 'carpet',
        '--cal-pro', 'False',
        '--sam-variant', sam,
        '--saliency-backbone', saliency,
        '--sam_checkpoint', ckpt,
        '--root-dir', f'./prof_{name}',
    ], cwd='/content/Segment-Any-Anomaly')
    print(f'{name} exit code:', r.returncode)

In [ ]:
# Gom ket qua profiling
import pandas as pd, glob, os

rows = {}
for p in sorted(glob.glob('./prof_*/csv/mvtec-indx-0.csv')):
    name = p.split('/')[1].replace('prof_', '')
    df = pd.read_csv(p, index_col=0)
    if 'carpet' in df.index:
        rows[name] = df.loc['carpet']

if rows:
    prof = pd.DataFrame(rows).T
    cols = [c for c in ['p_ap', 'p_f1', 't_dino', 't_sam', 't_saliency',
                        't_total', 'peak_vram', 'n_images'] if c in prof.columns]
    print(prof[cols].to_string(float_format='{:.2f}'.format))

    if 'baseline' in prof.index and 't_total' in prof.columns:
        base = prof.loc['baseline']
        print('\n--- so voi baseline (tieu chi spec muc 7) ---')
        for name in prof.index:
            if name == 'baseline':
                continue
            r = prof.loc[name]
            speedup = base['t_total'] / r['t_total'] if r['t_total'] else float('nan')
            print(f'{name:10s} speedup {speedup:5.2f}x  '
                  f'p_ap {r["p_ap"] / base["p_ap"] * 100:5.1f}%  '
                  f'p_f1 {r["p_f1"] / base["p_f1"] * 100:5.1f}%  '
                  f'(can >=3x va >=95%/95%)')
        print('\nt_dino/t_total:', {n: round(prof.loc[n, "t_dino"] / prof.loc[n, "t_total"], 3)
                                     for n in prof.index if prof.loc[n, 't_total']})
else:
    print('Chua co ket qua profiling nao.')

In [ ]:
%cd /content/Segment-Any-Anomaly
import pandas as pd, glob

def summarize(csv_path, label):
    df = pd.read_csv(csv_path, index_col=0)
    mean_row = df.mean(numeric_only=True).rename('MEAN')
    df = pd.concat([df, mean_row.to_frame().T])
    cols = [c for c in ['i_roc','p_roc','i_ap','p_ap','i_f1','p_f1',
                        'r_f1','r_f1_fixed','p_pro','t_dino','t_sam','t_saliency',
                        't_total','n_images','peak_vram'] if c in df.columns]
    print(f'\n{"="*60}\n  {label}\n{"="*60}')
    print(df[cols].to_string(float_format='{:.2f}'.format))

import os
root = os.environ.get('ROOT_DIR') or './result'

for p in glob.glob(f'{root}/csv/mvtec-indx-*.csv'):
    summarize(p, f'MVTec — {p}')
for p in glob.glob(f'{root}/csv/visa_public-indx-*.csv'):
    summarize(p, f'VisA — {p}')

# Doi chieu voi paper (docs/SAA+.md, bang SAA+):
#   MVTec-AD: max-F1-pixel 39.40, max-F1-region 49.67
#   VisA:     max-F1-pixel 27.07, max-F1-region 14.46
#
# HAI COT max-F1-region, khong lan lon:
#   r_f1       ban goc cua SAA+. Dung cot NAY de so voi 49.67, vi nhieu kha
#              nang paper tinh bang chinh doan code do. No CO THE VUOT 100
#              (class 'wood' ra 122.57): recall dem so vung DU DOAN khop
#              duoc roi chia cho so vung GT, nen nhieu manh du doan cung
#              trum mot vung GT la recall > 1.
#   r_f1_fixed cai dung dinh nghia ma chinh paper phat bieu (TP lay tu phep
#              ghep mot-mot). Luon <= 100. Day la so dung, nhung KHONG so
#              ngang voi bang da cong bo duoc.

In [ ]:
# Khong con buoc copy ra Drive: ROOT_DIR da nam san tren Drive, moi class
# chay xong la da ghi thang vao do. Cell nay chi de xac nhan.
import os, json, glob

root = os.environ['ROOT_DIR']
meta_path = f'{root}/csv/run_meta.json'

for p in sorted(glob.glob(f'{root}/csv/*')):
    print(f'{os.path.getsize(p):>10,} B  {p}')

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f'\n{len(meta)} class co metadata.')
    first = next(iter(meta.values()))
    print('GPU        :', first['gpu'])
    print('max_samples:', first['max_samples'], '(None = chay full)')
    print('cal_pro    :', first['cal_pro'])